# 🔍 Advanced Search & Indexing

This notebook demonstrates:
- Building and tuning **HNSW** indexes
- **BM25** keyword search (via `pg_textsearch`)
- **Ensemble search** (metadata + keyword + semantic)
- **Metadata operators** (`$gt`, `$in`, `$and`, `$or`, etc.)
- **Recall measurement** and **query plans**
- **Universal keyword search** across content + metadata

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from langchain_core.documents import Document
from pgvectordb import pgVectorDB, IndexType, KeywordSearchType, DistanceMetric, Config

In [2]:
rag = pgVectorDB(
    collection_name="nb_advanced",
    embedding_model=Config.get_embeddings(),
    connection_string=Config.get_connection_string(),
)
await rag.initialize(overwrite_existing=True)
print("✅ Ready")

✅ Ready


In [3]:
# Sample dataset: tech articles
docs = [
    Document(page_content="PostgreSQL 17 introduces incremental backup and improved COPY performance.",
             metadata={"topic": "database", "year": 2024, "author": "Alice"}),
    Document(page_content="pgvector 0.8 adds iterative scan and binary quantization support.",
             metadata={"topic": "database", "year": 2024, "author": "Bob"}),
    Document(page_content="DiskANN enables billion-scale vector search with memory optimization.",
             metadata={"topic": "database", "year": 2023, "author": "Alice"}),
    Document(page_content="Transformer models like BERT revolutionized natural language understanding.",
             metadata={"topic": "AI", "year": 2019, "author": "Carol"}),
    Document(page_content="RAG pipelines combine retrieval with LLM generation for grounded responses.",
             metadata={"topic": "AI", "year": 2024, "author": "Alice"}),
    Document(page_content="BM25 remains the gold standard for lexical information retrieval.",
             metadata={"topic": "AI", "year": 2020, "author": "Dave"}),
    Document(page_content="Kubernetes operators automate complex stateful application management.",
             metadata={"topic": "devops", "year": 2023, "author": "Bob"}),
    Document(page_content="Terraform infrastructure-as-code enables reproducible cloud deployments.",
             metadata={"topic": "devops", "year": 2022, "author": "Dave"}),
    Document(page_content="Cosine similarity measures the angle between embedding vectors.",
             metadata={"topic": "AI", "year": 2021, "author": "Carol"}),
    Document(page_content="Connection pooling with PgBouncer improves PostgreSQL scalability.",
             metadata={"topic": "database", "year": 2023, "author": "Carol"}),
]

ids = await rag.add_documents(docs)
print(f"✅ Added {len(ids)} documents")

✅ Added 10 documents


## 1. Build HNSW Index & Tune Parameters

In [4]:
await rag.build_index(
    metric=DistanceMetric.COSINE,
    m=16,               # max connections per node
    ef_construction=64,  # build-time candidate list
)
print("✅ HNSW index built")

# Tune query-time parameters
await rag.set_query_params(ef_search=100)
print("✅ ef_search set to 100")

✅ HNSW index built
✅ ef_search set to 100


In [5]:
# Check index stats
idx_stats = await rag.get_index_stats()
print("📊 Index Stats:")
for k, v in idx_stats.items():
    print(f"  {k}: {v}")

📊 Index Stats:
  index_type: hnsw
  index_built: True
  vector_size: 384
  indexes: [{'name': 'nb_advanced_pkey', 'definition': 'CREATE UNIQUE INDEX nb_advanced_pkey ON public.nb_advanced USING btree (langchain_id)'}, {'name': 'idx_nb_advanced_content_tsvector', 'definition': 'CREATE INDEX idx_nb_advanced_content_tsvector ON public.nb_advanced USING gin (content_tsvector)'}, {'name': 'idx_nb_advanced_content_trgm', 'definition': 'CREATE INDEX idx_nb_advanced_content_trgm ON public.nb_advanced USING gin (content gin_trgm_ops)'}, {'name': 'nb_advancedlangchainvectorindex', 'definition': "CREATE INDEX nb_advancedlangchainvectorindex ON public.nb_advanced USING hnsw (embedding vector_cosine_ops) WITH (m='16', ef_construction='64')"}]
  table_stats: {'inserts': 10, 'updates': 0, 'deletes': 0, 'live_tuples': 10, 'dead_tuples': 0, 'last_vacuum': None, 'last_autovacuum': None, 'last_analyze': None, 'last_autoanalyze': None, 'bloat_ratio': 0.0}
  size: {'total': '160 kB', 'table': '56 kB', 'ind

## 2. Advanced Metadata Filters

pgVectorDB supports MongoDB-style operators: `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$ne`, `$exists`, `$like`, `$ilike`, `$and`, `$or`.

In [6]:
# Year > 2022 AND topic = database
results = await rag.metadata_semantic_search(
    query="vector indexing",
    filter={
        "$and": [
            {"year": {"$gt": 2022}},
            {"topic": "database"}
        ]
    },
    k=3,
)
print("🎯 year>2022 AND topic=database:")
for r in results:
    print(f"  [{r['score']:.4f}] {r['content'][:70]}... ({r['metadata']})")

🎯 year>2022 AND topic=database:
  [0.5596] DiskANN enables billion-scale vector search with memory optimization.... ({'topic': 'database', 'year': 2023, 'author': 'Alice', 'langchain_id': 'b610e97f-6aa9-47b8-9704-fb3633f36096'})
  [0.5962] pgvector 0.8 adds iterative scan and binary quantization support.... ({'topic': 'database', 'year': 2024, 'author': 'Bob', 'langchain_id': 'ea6bdef3-221d-4026-9994-6af122f44b62'})
  [0.9118] Connection pooling with PgBouncer improves PostgreSQL scalability.... ({'topic': 'database', 'year': 2023, 'author': 'Carol', 'langchain_id': '05bc866a-e021-4374-9560-3d0888d99313'})


In [7]:
# Author IN [Alice, Bob]
results = await rag.metadata_filter(
    filter={"author": {"$in": ["Alice", "Bob"]}},
    k=5,
)
print(f"📋 Author in [Alice, Bob]: {len(results)} results")
for r in results:
    print(f"  {r['metadata']['author']}: {r['content'][:60]}...")

📋 Author in [Alice, Bob]: 5 results
  Alice: DiskANN enables billion-scale vector search with memory opti...
  Bob: pgvector 0.8 adds iterative scan and binary quantization sup...
  Alice: PostgreSQL 17 introduces incremental backup and improved COP...
  Alice: RAG pipelines combine retrieval with LLM generation for grou...
  Bob: Kubernetes operators automate complex stateful application m...


In [8]:
# Count matching docs
count = await rag.count_by_metadata(filter={"topic": "AI"})
print(f"📊 AI documents: {count}")

📊 AI documents: 4


## 3. Ensemble Search (Metadata + Hybrid)

In [9]:
results = await rag.ensemble_search(
    query="vector search database",
    filter={"year": {"$gte": 2023}},
    k=3,
    weights=(0.6, 0.4),  # semantic, keyword
)

print("🎼 Ensemble (year≥2023, semantic+keyword):")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content'][:70]}...")

🎼 Ensemble (year≥2023, semantic+keyword):
  1. [0.6000] DiskANN enables billion-scale vector search with memory optimization....
  2. [0.4068] pgvector 0.8 adds iterative scan and binary quantization support....
  3. [0.0903] Connection pooling with PgBouncer improves PostgreSQL scalability....


## 4. Universal Keyword Search (Content + Metadata)

In [10]:
results = await rag.universal_keyword_search(
    query="Alice",
    k=5,
    metadata_fields=["author"],  # also search inside metadata.author
)

print("🔤 Universal keyword 'Alice' (content + metadata):")
for r in results:
    print(f"  [{r['score']:.4f}] {r['content'][:60]}... (author: {r['metadata'].get('author')})")

🔤 Universal keyword 'Alice' (content + metadata):
  [0.0000] PostgreSQL 17 introduces incremental backup and improved COP... (author: Alice)
  [0.0000] DiskANN enables billion-scale vector search with memory opti... (author: Alice)
  [0.0000] RAG pipelines combine retrieval with LLM generation for grou... (author: Alice)


## 5. Recall Measurement

In [11]:
recall = await rag.compute_recall(
    test_queries=["vector search", "database indexing", "machine learning"],
    k=5,
)
print(f"📏 Recall Results:")
for k, v in recall.items():
    print(f"  {k}: {v}")

📏 Recall Results:
  recall@k: 1.0
  queries_tested: 3
  k: 5


## 6. Query Plan Analysis

In [12]:
plan = await rag.explain_query("vector search", search_method="semantic_search", k=3)
print("📋 Query Plan:")
for line in plan:
    print(f"  {line}")

📋 Query Plan:
  L
  i
  m
  i
  t
   
   
  (
  c
  o
  s
  t
  =
  3
  .
  2
  5
  .
  .
  3
  .
  2
  6
   
  r
  o
  w
  s
  =
  3
   
  w
  i
  d
  t
  h
  =
  8
  8
  )
   
  (
  a
  c
  t
  u
  a
  l
   
  t
  i
  m
  e
  =
  0
  .
  0
  3
  4
  .
  .
  0
  .
  0
  3
  6
   
  r
  o
  w
  s
  =
  3
   
  l
  o
  o
  p
  s
  =
  1
  )
  

   
   
  O
  u
  t
  p
  u
  t
  :
   
  l
  a
  n
  g
  c
  h
  a
  i
  n
  _
  i
  d
  ,
   
  c
  o
  n
  t
  e
  n
  t
  ,
   
  l
  a
  n
  g
  c
  h
  a
  i
  n
  _
  m
  e
  t
  a
  d
  a
  t
  a
  ,
   
  (
  (
  e
  m
  b
  e
  d
  d
  i
  n
  g
   
  <
  =
  >
   
  '
  [
  -
  0
  .
  0
  2
  0
  5
  5
  4
  1
  8
  ,
  0
  .
  0
  2
  8
  5
  6
  0
  2
  5
  7
  ,
  -
  0
  .
  0
  2
  3
  7
  1
  0
  9
  1
  8
  ,
  -
  0
  .
  0
  4
  8
  4
  4
  3
  0
  7
  5
  ,
  0
  .
  0
  4
  1
  2
  8
  8
  1
  6
  ,
  -
  0
  .
  0
  1
  3
  4
  9
  1
  0
  3
  5
  ,
  0
  .
  0
  1
  5
  4
  3
  9
  5
  9
  7
  ,
  -
  0
  .
  0
  6
  9
  

## 7. Collection Health Check

In [13]:
health = await rag.validate_collection()
print(f"🏥 Collection healthy: {health.get('is_healthy', 'N/A')}")
if health.get('issues'):
    for issue in health['issues']:
        print(f"  ⚠ {issue}")

🏥 Collection healthy: N/A


## 8. Export & Import

In [14]:
# Export to JSON
await rag.export_to_json("backup_advanced.json", include_embeddings=False)
print("💾 Exported to backup_advanced.json")

# Check file size
size = os.path.getsize("backup_advanced.json")
print(f"   File size: {size / 1024:.1f} KB")

💾 Exported to backup_advanced.json
   File size: 3.0 KB


In [15]:
await rag.delete_table()
await rag.close()
print("🧹 Cleaned up")

🧹 Cleaned up
